# 04 — Comprendre l'attention causale

**Objectif :** calculer comment chaque token rassemble l'information des tokens précédents pour former son contexte.

C'est la prochaine étape après les fenêtres glissantes : celles-ci fabriquent les séquences d'entraînement ; l'attention construit une représentation contextualisée de chaque position.

À la fin, tu sauras expliquer les requêtes (**Q**), les clés (**K**), les valeurs (**V**), le produit scalaire, le softmax et le masque causal. Nous utilisons de petits vecteurs et Python standard pour rendre chaque calcul visible. Aucun fichier de données n'est chargé et aucun modèle n'est entraîné.

## 1. Pourquoi le contexte ?

Dans « salam, labas ? », le sens d'un token dépend en partie des tokens autour de lui. L'attention calcule, pour chaque position, une combinaison pondérée des informations disponibles.

Dans un GPT, chaque token est d'abord représenté par un vecteur. Les matrices de projection Q, K et V sont apprises pendant l'entraînement. Pour comprendre le mécanisme, notre exemple utilise des vecteurs et des matrices fixés à la main : ils n'ont pas été appris sur la darija.

In [ ]:
tokens = ["salam", "kidayr", "labas", "?"]

# Vecteurs pédagogiques fictifs : une ligne par token, 3 nombres par vecteur.
embeddings = [
    [1.0, 0.0, 1.0],  # salam
    [0.0, 2.0, 1.0],  # kidayr
    [1.0, 1.0, 0.0],  # labas
    [0.0, 1.0, 2.0],  # ?
]

for token, vector in zip(tokens, embeddings):
    print(f"{token:>7} -> {vector}")

## 2. Requête, clé et valeur

Pour chaque token, trois projections produisent **Q**, **K** et **V**. Intuitivement, Q décrit ce que la position cherche, K ce que chaque position peut signaler, et V l'information qu'elle transmet.

On compare Q à toutes les K par produit scalaire. Après division par √dₖ et softmax, on obtient des poids qui totalisent 1. Ces poids font la moyenne pondérée des V. Le facteur √dₖ garde les scores à une échelle raisonnable quand les vecteurs grandissent.

In [ ]:
import math

def linear(vector, matrix):
    """Projeter un vecteur ligne par une matrice stockée par colonnes."""
    return [
        sum(value * matrix[row][column] for row, value in enumerate(vector))
        for column in range(len(matrix[0]))
    ]

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

def softmax(scores):
    peak = max(scores)
    exponents = [math.exp(score - peak) for score in scores]
    total = sum(exponents)
    return [value / total for value in exponents]

Pour isoler le calcul, les trois matrices ci-dessous sont des **poids fictifs** de dimensions 3 × 2. Dans un vrai Transformer, elles sont initialisées puis ajustées par la descente de gradient.

In [ ]:
W_q = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
W_k = [[1.0, 1.0], [1.0, 0.0], [0.0, 1.0]]
W_v = [[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]]

queries = [linear(x, W_q) for x in embeddings]
keys = [linear(x, W_k) for x in embeddings]
values = [linear(x, W_v) for x in embeddings]

print("Q[0] :", queries[0])
print("K[0] :", keys[0])
print("V[0] :", values[0])

## 3. Empêcher le regard vers le futur

Pendant la génération, le modèle doit prédire le token suivant sans connaître la réponse. Le **masque causal** interdit donc à la position i de consulter les positions j > i.

Avant le softmax, les scores futurs deviennent −∞. Leur poids après softmax vaut alors zéro. Le tableau ci-dessous affiche les poids d'attention ; une ligne correspond à la position qui regarde, une colonne à la position consultée.

In [ ]:
def causal_attention_weights(queries, keys):
    weights = []
    scale = math.sqrt(len(keys[0]))

    for i, query in enumerate(queries):
        scores = [dot(query, key) / scale for key in keys]
        masked_scores = [score if j <= i else float("-inf")
                         for j, score in enumerate(scores)]
        weights.append(softmax(masked_scores))

    return weights

weights = causal_attention_weights(queries, keys)
for token, row in zip(tokens, weights):
    print(f"{token:>7} -> {[round(weight, 3) for weight in row]}")

assert all(abs(sum(row) - 1.0) < 1e-9 for row in weights)
assert all(weights[i][j] == 0.0 for i in range(len(tokens))
           for j in range(i + 1, len(tokens)))

## 4. Mélanger les valeurs

Une fois les poids calculés, on les applique aux vecteurs V. La sortie à chaque position est son **vecteur de contexte** : une somme pondérée des valeurs accessibles.

La causalité se vérifie aussi en changeant le dernier token : les sorties des positions antérieures doivent rester identiques, car elles ne pouvaient déjà pas le consulter.

In [ ]:
def apply_attention(weights, values):
    outputs = []
    for row in weights:
        output = [
            sum(row[j] * values[j][dimension] for j in range(len(values)))
            for dimension in range(len(values[0]))
        ]
        outputs.append(output)
    return outputs

context_vectors = apply_attention(weights, values)
for token, vector in zip(tokens, context_vectors):
    print(f"{token:>7} -> {[round(value, 3) for value in vector]}")

changed_embeddings = [row[:] for row in embeddings]
changed_embeddings[-1] = [100.0, -50.0, 25.0]
changed_keys = [linear(x, W_k) for x in changed_embeddings]
changed_values = [linear(x, W_v) for x in changed_embeddings]
changed_weights = causal_attention_weights(queries, changed_keys)
changed_context = apply_attention(changed_weights, changed_values)

for i in range(len(tokens) - 1):
    assert all(abs(a - b) < 1e-9
               for a, b in zip(context_vectors[i], changed_context[i]))
print("Les positions passées ne changent pas quand le futur change.")

## 5. À retenir

- Q et K produisent les scores de compatibilité ; V transporte l'information à combiner.
- Le softmax transforme les scores en poids positifs de somme 1.
- La division par √dₖ met les scores à l'échelle avant le softmax.
- Le masque causal met à zéro l'influence des tokens futurs.
- Ici, toutes les valeurs et matrices sont fictives et fixes. Le tokenizer darija et le corpus ne sont pas utilisés ; nous n'avons pas encore de modèle entraîné.

Cette version en boucles sert à voir les opérations. La prochaine étape sera de réécrire le même calcul en PyTorch, puis d'ajouter plusieurs têtes d'attention et le bloc Transformer.

## Exercices

1. À la première position, combien de tokens peuvent recevoir un poids non nul ?
2. Pourquoi chaque ligne des poids somme-t-elle à 1 ?
3. Si l'on retire le masque, quel risque apparaît lors de l'entraînement pour prédire le prochain token ?
4. Quelle partie du calcul serait modifiée pour permettre à chaque tête d'attention de regarder une représentation différente ?

<details><summary>Voir les réponses</summary>

1. Un seul : la première position ne peut consulter que sa propre position.
2. Le softmax normalise les scores en une distribution de poids.
3. Une position pourrait accéder aux tokens cibles futurs, ce qui fuit la réponse attendue.
4. On utilise plusieurs ensembles de projections Q, K, V en parallèle, puis on combine leurs sorties : c'est l'attention multi-têtes.

</details>